In [4]:
def parse_scores(file):
    with open(file) as f:
        return [int(x) for x in f.read().split() if x.isdigit()]

human_scores = parse_scores("/Users/anilayhan/Desktop/Natural Computing/project last/NaCo/model/ais_human_out.txt")
llm_scores = parse_scores("/Users/anilayhan/Desktop/Natural Computing/project last/NaCo/model/ais_llm_out.txt")

# Detection thresholds (e.g., any non-zero score = detection)
human_detected = sum(1 for x in human_scores if x > 0)
llm_detected = sum(1 for x in llm_scores if x > 0)

human_total = len(human_scores)
llm_total = len(llm_scores)

# Equivalent to confusion matrix
TP = llm_detected         # correctly detected LLMs
FN = llm_total - TP
TN = human_total - human_detected
FP = human_detected

accuracy = (TP + TN) / (human_total + llm_total)


In [5]:
print(accuracy)

0.47661757038581853


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

np.random.seed(42)

# === CONFIGURATION ===
LLM_FILE = "/Users/anilayhan/Desktop/Natural Computing/project last/NaCo/stats/outputasd2.txt"
HUMAN_FILE = "/Users/anilayhan/Desktop/Natural Computing/project last/NaCo/our_data/train_human_clean.txt"

# Load data
with open(HUMAN_FILE) as f:
    human_lines = [line.strip() for line in f if line.strip()]
with open(LLM_FILE) as f:
    llm_lines = [line.strip() for line in f if line.strip()]

# Downsample human text to match LLM count
if len(human_lines) > len(llm_lines):
    sampled_idx = np.random.choice(len(human_lines), size=len(llm_lines), replace=False)
    human_lines = [human_lines[i] for i in sampled_idx]

# Labels
human_labels = [0] * len(human_lines)
llm_labels = [1] * len(llm_lines)

# Combine data
all_texts = human_lines + llm_lines
all_labels = human_labels + llm_labels

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    all_texts,
    all_labels,
    test_size=0.3,
    random_state=42,
    stratify=all_labels
)

# TF-IDF
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Logistic regression
clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
clf.fit(X_train_vec, y_train)

# Predictions
y_pred = clf.predict(X_test_vec)
y_pred_proba = clf.predict_proba(X_test_vec)

# Metrics
acc = accuracy_score(y_test, y_pred)
conf = confusion_matrix(y_test, y_pred)

results_summary = {
    "Accuracy": acc,
    "Human Detection Rate": conf[0, 0] / (conf[0, 0] + conf[0, 1]),
    "LLM Detection Rate": conf[1, 1] / (conf[1, 0] + conf[1, 1]),
    "Confusion Matrix": conf.tolist()
}

print(results_summary)


{'Accuracy': 0.9379932356257046, 'Human Detection Rate': np.float64(0.9166040570999249), 'LLM Detection Rate': np.float64(0.9593984962406015), 'Confusion Matrix': [[1220, 111], [54, 1276]]}


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
import numpy as np

# Load data
with open("/Users/anilayhan/Desktop/Natural Computing/project last/NaCo/our_data/train_human_clean.txt") as f:
    human_lines = [line.strip() for line in f if line.strip()]
with open("/Users/anilayhan/Desktop/Natural Computing/project last/NaCo/our_data/test_llm_clean.txt") as f:
    llm_lines = [line.strip() for line in f if line.strip()]

# Labels
human_labels = [0] * len(human_lines)
llm_labels = [1] * len(llm_lines)

# Data
X = human_lines + llm_lines
y = human_labels + llm_labels

# TF-IDF
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)
X_vec = vectorizer.fit_transform(X)

# Train classifier
clf = LogisticRegression(max_iter=1000)
clf.fit(X_vec, y)
y_pred = clf.predict(X_vec)

# Evaluation
acc = accuracy_score(y, y_pred)
conf = confusion_matrix(y, y_pred)

print(f"Accuracy: {acc:.3f}")
print("Confusion Matrix:")
print(conf)
print(f"Human Detection Rate: {conf[0,0] / (conf[0,0] + conf[0,1]):.3f}")
print(f"LLM Detection Rate: {conf[1,1] / (conf[1,0] + conf[1,1]):.3f}")


Accuracy: 0.997
Confusion Matrix:
[[762965     46]
 [  2425   1764]]
Human Detection Rate: 1.000
LLM Detection Rate: 0.421


Accuracy: 0.997
Confusion Matrix:
[[762965     46]
 [  2425   1764]]
Human Detection Rate: 1.000
LLM Detection Rate: 0.421


In [18]:
import numpy as np

# Load reactivity scores
human_scores = [int(x) for x in open("/Users/anilayhan/Desktop/Natural Computing/project last/NaCo/model/reactivity_human0.txt") if x.strip().isdigit()]
llm_scores = [int(x) for x in open("/Users/anilayhan/Desktop/Natural Computing/project last/NaCo/model/reactivity_llm0.txt") if x.strip().isdigit()]

# Labels: 0 for human, 1 for LLM
y_true = [0] * len(human_scores) + [1] * len(llm_scores)
y_pred = []

# You can choose a simple threshold. Try: score > 0 ⇒ predicted as LLM
for score in human_scores:
    y_pred.append(1 if score > 0 else 0)
for score in llm_scores:
    y_pred.append(1 if score > 0 else 0)

from sklearn.metrics import accuracy_score, confusion_matrix

acc = accuracy_score(y_true, y_pred)
conf = confusion_matrix(y_true, y_pred)

print("Accuracy:", acc)
print("Confusion Matrix:")
print(conf)


Accuracy: 0.5372940563086549
Confusion Matrix:
[[412212 350799]
 [  4189      0]]
